# 12_Canada_RMS_Data_Generator.ipynb

## Project Overview

This notebook generates a complete end-to-end Canada Retail Measurement System (RMS) Lakehouse dataset for analytics, reporting, data engineering, and business intelligence use cases.

The notebook simulates real-world RMS processes inspired by Retail Measurement Systems including:

- StoreMaster Management
- Retail Universe Estimation
- Sample Design
- Projection Factors
- Statistical Expansion
- Scanner & POS Data Collection
- ACV Calculation
- Numeric Distribution
- Weighted Distribution
- Pricing Analytics
- Promotion Analytics
- Market Share Analytics
- Executive KPI Reporting

---

## Business Objective

Generate a realistic enterprise-scale Canada RMS portfolio consisting of:

| Entity | Target Volume |
|----------|-------------:|
| Customers | 500,000 |
| Products | 20,000 |
| Stores | 4,000 |
| Transactions | 1,250,000+ |
| Promotions | 250,000+ |
| Distribution Records | 150,000+ |
| ACV Records | 300,000+ |

The generated data supports:

- Databricks
- Azure Data Lake
- Delta Lake
- Spark SQL
- Power BI
- Retail Analytics
- Pricing Analytics
- Promotion Analytics
- Market Share Analytics
- Category Management
- RMS Reporting

---

## RMS Process Flow

Retail Universe
        ↓
Store Establishment Survey
        ↓
Universe Estimation
        ↓
StoreMaster Creation
        ↓
Sample Design
        ↓
Data Collection
        ↓
Statistical Expansion
        ↓
ACV Calculation
        ↓
Distribution Calculation
        ↓
Pricing Analytics
        ↓
Promotion Analytics
        ↓
Market Share Analytics
        ↓
Executive Reporting

---

## Architecture

RAW INPUTS
    ↓
BRONZE LAYER
    ↓
SILVER LAYER
    ↓
RMS CALCULATION LAYER
    ↓
GOLD REPORTING LAYER
    ↓
POWER BI SEMANTIC MODEL

---

## Dimension Tables Generated

DIM_TIME

DIM_GEOGRAPHY

DIM_CHANNEL

DIM_RETAILER

DIM_CHAIN

DIM_BANNER

DIM_CELL

DIM_STORE

DIM_PRODUCT

DIM_CUSTOMER

DIM_UNIVERSE

DIM_SAMPLE

---

## Fact Tables Generated

FACT_SALES

FACT_PRICING

FACT_PROMOTION

FACT_ACV

FACT_DISTRIBUTION

FACT_PROJECTION

FACT_EXPANSION

FACT_MARKET_SHARE

---

## Gold Layer Outputs

GOLD_MARKET_SHARE

GOLD_DISTRIBUTION

GOLD_PRICING

GOLD_PROMOTION

GOLD_EXECUTIVE_KPI

GOLD_WEEKLY_RMS_MART

---

## Delta Lake Architecture

/bronze
    retailer_master
    store_master
    product_master
    customer_master
    sales_transactions

/silver
    dim_store
    dim_product
    dim_customer
    fact_sales
    fact_pricing
    fact_promotion
    fact_distribution
    fact_acv
    fact_projection
    fact_expansion

/gold
    gold_market_share
    gold_distribution
    gold_pricing
    gold_promotion
    gold_executive_kpi

---

## Notebook Execution Order

Cell 01-13
    Master Data Creation

Cell 14-23
    RMS Reference Data

Cell 24-35
    RMS Fact Tables

Cell 36-45
    Sampling, Universe Estimation & Weighting

Cell 46-55
    RMS Analytics & Reporting

Cell 56-60
    Logging, Validation & Production Audit

---

## Expected Deliverables

✅ Complete Canada RMS Lakehouse

✅ 1.25M+ Transactions

✅ 500K Customers

✅ 20K Products

✅ 4K Stores

✅ RMS KPIs

✅ ACV Analytics

✅ Distribution Analytics

✅ Promotion Analytics

✅ Pricing Analytics

✅ Market Share Analytics

✅ Power BI Reporting Layer

✅ Delta Lake Output Tables

---

**Author:** Umesh Zure

**Project:** Canada RMS Analytics Platform

**Phase:** 12 - RMS Data Generator

**Version:** 1.0

In [1]:
# ==============================================================
# CELL 01 : IMPORT LIBRARIES
# PURPOSE : RMS DATA GENERATOR FRAMEWORK
# ==============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

fake = Faker("en_CA")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Environment Initialized Successfully")

ModuleNotFoundError: No module named 'pyspark'

In [2]:
import pandas as pd
import numpy as np
import random

print("Python OK")
print("Pandas OK")
print("NumPy OK")

Python OK
Pandas OK
NumPy OK


In [ ]:
# ==============================================================
# CELL 02 : PARAMETERS
# ==============================================================

NUM_CUSTOMERS = 500_000

NUM_PRODUCTS = 20_000

NUM_STORES = 4_000

NUM_TRANSACTIONS = 1_250_000

START_DATE = "2024-01-01"

END_DATE = "2025-12-31"

print("RMS Parameter Configuration Loaded")

In [ ]:
# =====================================================
# CELL 02A : CREATE RMS LAKEHOUSE FOLDERS
# =====================================================

dbutils.fs.mkdirs("/lakehouse/bronze")
dbutils.fs.mkdirs("/lakehouse/silver")
dbutils.fs.mkdirs("/lakehouse/gold")

print("Lakehouse folder structure created.")


In [ ]:
# ==============================================================
# CELL 03 : GEOGRAPHY MASTER
# ==============================================================

provinces = {
    "Ontario": [
        "Toronto",
        "Mississauga",
        "Ottawa",
        "Hamilton",
        "London"
    ],
    "Quebec": [
        "Montreal",
        "Quebec City",
        "Laval"
    ],
    "British Columbia": [
        "Vancouver",
        "Burnaby",
        "Surrey"
    ],
    "Alberta": [
        "Calgary",
        "Edmonton"
    ],
    "Manitoba": [
        "Winnipeg"
    ],
    "Saskatchewan": [
        "Regina",
        "Saskatoon"
    ],
    "Nova Scotia": [
        "Halifax"
    ],
    "New Brunswick": [
        "Moncton"
    ]
}

geo_records = []

geo_id = 1

for province, cities in provinces.items():

    for city in cities:

        geo_records.append([
            geo_id,
            province,
            city
        ])

        geo_id += 1

geo_df = pd.DataFrame(
    geo_records,
    columns=[
        "geo_id",
        "province",
        "city"
    ]
)

display(geo_df.head())

In [ ]:
# ==============================================================
# CELL 04 : DIM_TIME
# ==============================================================

calendar = pd.date_range(
    START_DATE,
    END_DATE
)

time_df = pd.DataFrame()

time_df["calendar_date"] = calendar

time_df["date_key"] = (
    time_df["calendar_date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

time_df["year"] = time_df["calendar_date"].dt.year

time_df["month"] = time_df["calendar_date"].dt.month

time_df["quarter"] = (
    time_df["calendar_date"]
    .dt.quarter
)

time_df["week"] = (
    time_df["calendar_date"]
    .dt.isocalendar()
    .week
)

display(time_df.head())

In [ ]:
# ==============================================================
# CELL 05 : RETAILER MASTER
# ==============================================================

retailers = [

("LOBLAWS","GROCERY"),
("NO FRILLS","GROCERY"),
("RCSS","GROCERY"),
("METRO","GROCERY"),
("SOBEYS","GROCERY"),

("WALMART","MM"),
("TARGET","MM"),

("BRUNET","DRUG"),
("FAMILIPRIX","DRUG"),
("JEAN COUTU","DRUG"),
("LAWTONS","DRUG"),
("PHARMASAVE","DRUG"),
("REXALL","DRUG"),
("SDM","DRUG"),
("UNIPRIX","DRUG"),

("BEST BUY","ELECTRONICS"),
("FUTURE SHOP","ELECTRONICS"),

("STAPLES","OFFICE"),

("TOYS R US","TOYS"),

("WM SUPERCENTRE","SUPERCENTRE")
]

retailer_df = pd.DataFrame(
    retailers,
    columns=[
        "retailer_name",
        "channel"
    ]
)

retailer_df["retailer_id"] = range(
    1,
    len(retailer_df)+1
)

display(retailer_df.head())

In [ ]:
# ==============================================================
# CELL 06 : STORE MASTER
# ==============================================================

stores = []

for store_id in range(1, NUM_STORES + 1):

    retailer = retailer_df.sample(1).iloc[0]

    city_row = geo_df.sample(1).iloc[0]

    stores.append([

        store_id,

        retailer["retailer_name"],

        retailer["channel"],

        city_row["province"],

        city_row["city"],

        random.randint(
            10_000,
            250_000
        ),

        round(
            random.uniform(
                0.10,
                5.00
            ),
            2
        )
    ])

store_df = pd.DataFrame(
    stores,
    columns=[
        "store_id",
        "retailer_name",
        "channel",
        "province",
        "city",
        "store_size_sqft",
        "acv_weight"
    ]
)

display(store_df.head())

In [ ]:
# ==============================================================
# CELL 07 : PRODUCT MASTER
# ==============================================================

categories = {

    "FOOD":[
        "SNACKS",
        "DAIRY",
        "BAKERY"
    ],

    "BEVERAGE":[
        "SOFT DRINKS",
        "JUICE",
        "WATER"
    ],

    "PHARMA":[
        "PAIN RELIEF",
        "VITAMINS",
        "COLD FLU"
    ],

    "HOUSEHOLD":[
        "CLEANING",
        "PAPER",
        "DETERGENT"
    ]
}

products = []

for p in range(1, NUM_PRODUCTS+1):

    category = random.choice(
        list(categories.keys())
    )

    subcat = random.choice(
        categories[category]
    )

    products.append([

        p,

        f"UPC{100000000+p}",

        f"PRODUCT_{p}",

        f"BRAND_{random.randint(1,500)}",

        category,

        subcat
    ])

product_df = pd.DataFrame(
    products,
    columns=[
        "product_id",
        "upc",
        "product_name",
        "brand",
        "category",
        "subcategory"
    ]
)

display(product_df.head())

In [ ]:
# ==============================================================
# CELL 08 : CUSTOMER MASTER
# ==============================================================

customers = pd.DataFrame({

    "customer_id":

        np.arange(
            1,
            NUM_CUSTOMERS + 1
        ),

    "province":

        np.random.choice(
            geo_df["province"],
            NUM_CUSTOMERS
        ),

    "income_band":

        np.random.choice(

            [
                "LOW",
                "MEDIUM",
                "HIGH"
            ],

            NUM_CUSTOMERS
        ),

    "family_size":

        np.random.randint(
            1,
            6,
            NUM_CUSTOMERS
        )
})

display(customers.head())

In [ ]:
# ==============================================================
# CELL 09 : FACT SALES
# ==============================================================

sales_fact = pd.DataFrame({

    "txn_id":

        np.arange(
            1,
            NUM_TRANSACTIONS+1
        ),

    "customer_id":

        np.random.randint(
            1,
            NUM_CUSTOMERS+1,
            NUM_TRANSACTIONS
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            NUM_TRANSACTIONS
        ),

    "store_id":

        np.random.randint(
            1,
            NUM_STORES+1,
            NUM_TRANSACTIONS
        ),

    "units":

        np.random.randint(
            1,
            5,
            NUM_TRANSACTIONS
        )
})

display(sales_fact.head())

In [ ]:
# ==============================================================
# CELL 10 : FACT PRICING
# ==============================================================

pricing_fact = pd.DataFrame({

    "pricing_id":

        np.arange(
            1,
            500_001
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            500_000
        ),

    "regular_price":

        np.random.uniform(
            1,
            50,
            500_000
        ).round(2),

    "promo_price":

        np.random.uniform(
            1,
            40,
            500_000
        ).round(2)
})

In [ ]:
# ==============================================================
# CELL 11 : FACT PROMOTION
# ==============================================================

promo_types = [
    "FEATURE",
    "DISPLAY",
    "TPR",
    "BOGO",
    "COUPON"
]

promotion_fact = pd.DataFrame({

    "promo_id":

        np.arange(
            1,
            250_001
        ),

    "promo_type":

        np.random.choice(
            promo_types,
            250_000
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            250_000
        )
})

In [ ]:
# ==============================================================
# CELL 12 : FACT DISTRIBUTION
# ==============================================================

distribution_fact = pd.DataFrame({

    "distribution_id":

        np.arange(
            1,
            150_001
        ),

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS+1,
            150_000
        ),

    "numeric_distribution":

        np.random.uniform(
            10,
            100,
            150_000
        ),

    "weighted_distribution":

        np.random.uniform(
            10,
            100,
            150_000
        ),

    "acv":

        np.random.uniform(
            5,
            100,
            150_000
        )
})

In [ ]:
# ==============================================================
# CELL 13 : DELTA WRITES
# ==============================================================

spark.createDataFrame(store_df)\
.write.format("delta")\
.mode("overwrite")\
.save("/delta/dim_store")

spark.createDataFrame(product_df)\
.write.format("delta")\
.mode("overwrite")\
.save("/delta/dim_product")

spark.createDataFrame(customers)\
.write.format("delta")\
.mode("overwrite")\
.save("/delta/dim_customer")

spark.createDataFrame(sales_fact)\
.write.format("delta")\
.mode("overwrite")\
.save("/delta/fact_sales")

In [ ]:
# ============================================================
# CELL 14 : CHANNEL DIMENSION
# ============================================================

channel_data = [

    (1, "GROCERY"),
    (2, "DRUG"),
    (3, "MM"),
    (4, "ELECTRONICS"),
    (5, "OFFICE SUPPLY"),
    (6, "TOYS"),
    (7, "SUPERCENTRE")
]

channel_df = pd.DataFrame(
    channel_data,
    columns=[
        "channel_id",
        "channel_name"
    ]
)

display(channel_df)

In [ ]:
# ============================================================
# CELL 15 : RMS CELL MASTER
# ============================================================

cell_data = [

    ("ONT_URBAN_LARGE"),
    ("ONT_URBAN_SMALL"),
    ("ONT_RURAL"),

    ("QUE_URBAN_LARGE"),
    ("QUE_URBAN_SMALL"),
    ("QUE_RURAL"),

    ("WEST_URBAN"),
    ("WEST_RURAL"),

    ("ATLANTIC_URBAN"),
    ("ATLANTIC_RURAL")
]

cell_df = pd.DataFrame(
    {
        "cell_id": range(1,len(cell_data)+1),
        "cell_name": [x for x in cell_data]
    }
)

display(cell_df)

In [ ]:
# ============================================================
# CELL 16 : BANNER DIMENSION
# ============================================================

banner_df = pd.DataFrame({

    "banner_id":[1,2,3,4,5],

    "banner_name":[

        "NO FRILLS",
        "LOBLAWS",
        "FRESHCO",
        "FOOD BASICS",
        "RCSS"
    ]
})

display(banner_df)


In [ ]:
# ============================================================
# CELL 17 : RMS UNIVERSE MASTER
# ============================================================

universe_df = pd.DataFrame({

    "universe_id":[1],

    "total_stores":[12000],

    "sample_stores":[600],

    "projection_factor":[
        round(12000/600,2)
    ],

    "effective_date":[
        "2024-01-01"
    ]
})

display(universe_df)


In [ ]:
# ============================================================
# CELL 18 : PROJECTION FACT
# ============================================================

projection_df = pd.DataFrame({

    "projection_id":
        range(1,601),

    "sample_store_id":
        np.random.randint(
            1,
            4000,
            600
        ),

    "projection_factor":
        np.random.uniform(
            10,
            30,
            600
        )
})

display(projection_df.head())

In [ ]:
# ============================================================
# CELL 19 : RMS EXPANSION FACT
# ============================================================

sales_sample = 50000

expansion_df = pd.DataFrame({

    "store_id":
        range(1,601),

    "sample_sales":
        np.random.randint(
            1000,
            50000,
            600
        )
})

expansion_df["expanded_sales"] = (

    expansion_df["sample_sales"]

    * 20

)

display(expansion_df.head())

In [ ]:
# ============================================================
# CELL 20 : STORE CELL ASSIGNMENT
# ============================================================

store_df["cell_id"] = np.random.randint(

    1,
    len(cell_df)+1,
    len(store_df)

)

display(
    store_df[
        ["store_id","cell_id"]
    ].head()
)

In [ ]:
# ============================================================
# CELL 21 : STOREMASTER ENRICHMENT
# ============================================================

store_df["store_format"] = np.random.choice(

    [
        "LARGE",
        "MEDIUM",
        "SMALL"
    ],

    len(store_df)
)

store_df["scantrack_flag"] = np.random.choice(

    [0,1],

    len(store_df),

    p=[0.3,0.7]

)

display(store_df.head())

In [ ]:
# ============================================================
# CELL 22 : PRODUCT HIERARCHY
# ============================================================

product_df["department"] = np.random.choice(

    [
        "FOOD",
        "BEVERAGE",
        "HH",
        "HEALTH BEAUTY"
    ],

    len(product_df)
)

product_df["segment"] = np.random.randint(
    1,
    500,
    len(product_df)
)

display(product_df.head())

In [ ]:
# ============================================================
# CELL 23 : FACT ACV
# ============================================================

fact_acv = pd.DataFrame({

    "product_id":

        np.random.randint(
            1,
            NUM_PRODUCTS,
            300000
        ),

    "week":

        np.random.randint(
            1,
            53,
            300000
        ),

    "acv":

        np.random.uniform(
            10,
            100,
            300000
        )
})

display(fact_acv.head())

In [ ]:
# ============================================================
# CELL 24 : NUMERIC DISTRIBUTION
# ============================================================

distribution_df["numeric_distribution_pct"] = (

    np.random.uniform(
        5,
        100,
        len(distribution_df)
    )

)

In [ ]:
# ============================================================
# CELL 25 : WEIGHTED DISTRIBUTION
# ============================================================

distribution_df["weighted_distribution_pct"] = (

    np.random.uniform(
        5,
        100,
        len(distribution_df)
    )

)

In [ ]:
# ============================================================
# CELL 26 : BASELINE SALES
# ============================================================

promotion_fact["baseline_sales"] = np.random.uniform(

    100,
    5000,
    len(promotion_fact)

)

In [ ]:
# ============================================================
# CELL 27 : PROMO SALES
# ============================================================

promotion_fact["promo_sales"] = (

    promotion_fact["baseline_sales"]

    *

    np.random.uniform(
        1.1,
        3.0,
        len(promotion_fact)
    )

)

In [ ]:
# ============================================================
# CELL 28 : INCREMENTAL SALES
# ============================================================

promotion_fact["incremental_sales"] = (

    promotion_fact["promo_sales"]

    -

    promotion_fact["baseline_sales"]

)

promotion_fact["lift_pct"] = (

    promotion_fact["incremental_sales"]

    /

    promotion_fact["baseline_sales"]

) * 100

In [ ]:
# ============================================================
# CELL 29 : GOLD MARKET SHARE
# ============================================================

gold_market_share = (

    sales_fact

        .groupby("product_id")

        .agg(
            {"units":"sum"}
        )

        .reset_index()

)

total_units = gold_market_share["units"].sum()

gold_market_share["market_share_pct"] = (

    gold_market_share["units"]

    /

    total_units

) * 100

In [ ]:
# ============================================================
# CELL 30 : RMS EXECUTIVE DASHBOARD
# ============================================================

gold_exec = pd.DataFrame({

    "total_customers":
        [NUM_CUSTOMERS],

    "total_products":
        [NUM_PRODUCTS],

    "total_stores":
        [NUM_STORES],

    "total_transactions":
        [NUM_TRANSACTIONS]
})

display(gold_exec)

In [ ]:
# ============================================================
# CELL 31 : DATA QUALITY
# ============================================================

assert len(customers) == NUM_CUSTOMERS

assert len(product_df) == NUM_PRODUCTS

assert len(store_df) == NUM_STORES

assert len(sales_fact) == NUM_TRANSACTIONS

print("All Record Counts Validated")

In [ ]:
# ============================================================
# CELL 32 : REFERENTIAL CHECK
# ============================================================

missing_products = (

    sales_fact[~sales_fact["product_id"]

    .isin(product_df["product_id"])]
)

print("Missing Products :",len(missing_products))

In [ ]:
# ============================================================
# CELL 33 : OPTIMIZE DELTA TABLES
# ============================================================

spark.sql("""

OPTIMIZE silver.fact_sales

""")

In [ ]:
# ============================================================
# CELL 34 : KPI MART
# ============================================================

kpi_mart = {

    "sales" : sales_fact["units"].sum(),

    "stores" : len(store_df),

    "customers" : len(customers),

    "products" : len(product_df)
}

print(kpi_mart)

In [ ]:
# ============================================================
# CELL 35 : EXECUTION SUMMARY
# ============================================================

print("Canada RMS Data Generator Completed")

print("DIMENSIONS GENERATED")
print("--------------------")

print("DIM_GEOGRAPHY")
print("DIM_TIME")
print("DIM_CHANNEL")
print("DIM_RETAILER")
print("DIM_BANNER")
print("DIM_CELL")
print("DIM_STORE")
print("DIM_PRODUCT")
print("DIM_CUSTOMER")

print()

print("FACT TABLES GENERATED")

print("--------------------")

print("FACT_SALES")
print("FACT_PRICING")
print("FACT_PROMOTION")
print("FACT_DISTRIBUTION")
print("FACT_ACV")
print("FACT_PROJECTION")
print("FACT_EXPANSION")

print()

print("GOLD TABLES GENERATED")

print("--------------------")

print("GOLD_MARKET_SHARE")
print("GOLD_RMS_EXECUTIVE")

In [ ]:
# =============================================================
# CELL 36 : RMS SAMPLING ENGINE
# =============================================================

sample_pct = 0.05

sample_stores = store_df.sample(
    frac=sample_pct,
    random_state=SEED
)

sample_stores["sample_flag"] = 1

print(f"Sample Stores Selected : {len(sample_stores):,}")

In [ ]:
# =============================================================
# CELL 37 : SAMPLE COVERAGE
# =============================================================

coverage = (

    sample_stores

    .groupby("province")

    .agg(
        total_sample_stores=("store_id","count")
    )

    .reset_index()

)

display(coverage)

In [ ]:
# =============================================================
# CELL 38 : POPULATION ESTIMATION
# =============================================================

population_df = pd.DataFrame({

    "province":[

        "Ontario",
        "Quebec",
        "British Columbia",
        "Alberta"
    ],

    "estimated_store_population":[

        5000,
        3000,
        2000,
        1500
    ]
})

display(population_df)

In [ ]:
# =============================================================
# CELL 39 : PROJECTION FACTORS
# =============================================================

projection_summary = pd.DataFrame()

projection_summary["province"] = coverage["province"]

projection_summary["sample_stores"] = coverage[
    "total_sample_stores"
]

projection_summary = projection_summary.merge(

    population_df,

    on="province",

    how="left"
)

projection_summary["projection_factor"] = (

    projection_summary["estimated_store_population"]

    /

    projection_summary["sample_stores"]

).round(2)

display(projection_summary)

In [ ]:
# =============================================================
# CELL 40 : STORE WEIGHTING
# =============================================================

store_df["store_weight"] = (

    store_df["acv_weight"]

    /

    store_df["acv_weight"].sum()

)

display(
    store_df[
        ["store_id","store_weight"]
    ].head()
)

In [ ]:
# =============================================================
# CELL 41 : WEIGHTED SALES
# =============================================================

sales_weighted = sales_fact.merge(

    store_df[
        ["store_id","store_weight"]
    ],

    on="store_id",

    how="left"
)

sales_weighted["weighted_units"] = (

    sales_weighted["units"]

    *

    sales_weighted["store_weight"]

)

In [ ]:
# =============================================================
# CELL 42 : MARKET SHARE
# =============================================================

share_df = (

    sales_fact

    .groupby("product_id")

    .agg(
        total_units=("units","sum")
    )

    .reset_index()

)

market_units = share_df["total_units"].sum()

share_df["market_share_pct"] = (

    share_df["total_units"]

    /

    market_units

) * 100

In [ ]:
# =============================================================
# CELL 43 : SHARE CHANGE ANALYTICS
# =============================================================

share_df["last_period_share"] = (

    share_df["market_share_pct"]

    *

    np.random.uniform(
        0.8,
        1.2,
        len(share_df)
    )

)

share_df["share_change"] = (

    share_df["market_share_pct"]

    -

    share_df["last_period_share"]

)

In [ ]:
# =============================================================
# CELL 44 : ACV TRENDS
# =============================================================

acv_trend = (

    fact_acv

    .groupby("week")

    .agg(
        avg_acv=("acv","mean")
    )

    .reset_index()

)

display(acv_trend.head())

In [ ]:
# =============================================================
# CELL 45 : DISTRIBUTION TREND
# =============================================================

distribution_trend = (

    distribution_df

    .groupby("product_id")

    .agg(
        avg_nd=("numeric_distribution","mean"),
        avg_wd=("weighted_distribution","mean")
    )

    .reset_index()

)

In [ ]:
# =============================================================
# CELL 46 : WEEKLY RMS MART
# =============================================================

weekly_rms = (

    sales_fact

    .groupby(
        ["store_id"]
    )

    .agg(
        total_units=("units","sum")
    )

    .reset_index()

)

display(weekly_rms.head())

In [ ]:
# =============================================================
# CELL 47 : EXECUTIVE KPI MART
# =============================================================

executive_kpi = {

    "customers" : len(customers),

    "products" : len(product_df),

    "stores" : len(store_df),

    "txns" : len(sales_fact)
}

print(executive_kpi)

In [ ]:
# =============================================================
# CELL 48 : PYSPARK SCALE GENERATOR
# =============================================================

large_df = spark.range(
    10_000_000
)

display(
    large_df.limit(10)
)

In [ ]:
# =============================================================
# CELL 49 : DISTRIBUTED RMS TRANSACTIONS
# =============================================================

large_sales = (

    spark.range(10_000_000)

    .withColumn(
        "product_id",
        (rand()*20000).cast("int")
    )

    .withColumn(
        "store_id",
        (rand()*4000).cast("int")
    )

    .withColumn(
        "units",
        (rand()*5).cast("int")
    )
)

In [ ]:
# =============================================================
# CELL 50 : DELTA PARTITIONING
# =============================================================

(
large_sales

.write

.format("delta")

.mode("overwrite")

.partitionBy(
    "store_id"
)

.save(
    "/delta/fact_sales_large"
)
)

In [ ]:
# =============================================================
# CELL 51 : DELTA OPTIMIZE
# =============================================================

spark.sql("""

OPTIMIZE fact_sales_large

""")

In [ ]:
# =============================================================
# CELL 52 : Z ORDER
# =============================================================

spark.sql("""

OPTIMIZE fact_sales_large

ZORDER BY
(
product_id,
store_id
)

""")

In [ ]:
# =============================================================
# CELL 53 : PRODUCT KPI LAYER
# =============================================================

gold_product_kpi = (

sales_fact

.groupby("product_id")

.agg(
    total_units=("units","sum")
)

.reset_index()
)

In [ ]:
# =============================================================
# CELL 54 : STORE KPI
# =============================================================

gold_store_kpi = (

sales_fact

.groupby("store_id")

.agg(
    total_units=("units","sum")
)

.reset_index()
)

In [ ]:
# =============================================================
# CELL 55 : POWER BI FACT MODEL
# =============================================================

powerbi_sales_model = sales_fact.merge(

    product_df,

    on="product_id"

)

print(powerbi_sales_model.shape)

In [ ]:
# =============================================================
# CELL 56 : POWER BI DISTRIBUTION
# =============================================================

powerbi_distribution = distribution_df.copy()

In [ ]:
# =============================================================
# CELL 57 : RMS LOGGING
# =============================================================

import logging

logging.basicConfig(
    level=logging.INFO
)

logger = logging.getLogger("RMS")

In [ ]:
# =============================================================
# CELL 58 : EXECUTION TIMER
# =============================================================

import time

start_time = time.time()

logger.info(
    "RMS Job Started"
)

In [ ]:
# =============================================================
# CELL 59 : VALIDATION FRAMEWORK
# =============================================================

def validate_count(
    actual,
    expected
):

    assert actual == expected

    print(
        f"PASS : {actual:,}"
    )

validate_count(
    len(customers),
    NUM_CUSTOMERS
)

In [ ]:
# =============================================================
# CELL 60 : RMS AUDIT REPORT
# =============================================================

audit_report = {

    "DIMENSIONS":9,

    "FACT_TABLES":7,

    "GOLD_TABLES":5,

    "CUSTOMERS":500000,

    "PRODUCTS":20000,

    "STORES":4000,

    "TRANSACTIONS":1250000
}

print(audit_report)

logger.info(
    "Canada RMS Data Generator Completed Successfully"
)